# PPE zero-shot VLM evaluation — no Pandas needed

This edition avoids **Pandas, NumPy, Matplotlib and scikit-learn**, which can require native DLL files blocked by some Windows Application Control settings. It relies on Python's standard libraries for CSV processing and metrics; the Ollama client and Jupyter/IPython are still required for model requests and notebook display.

- Compare your **human-verified image labels** to VLM predictions (zero-shot; **no training**).
- Measure **accuracy, precision, recall, F1, false-positive rate, prediction coverage, strict accuracy**, and processing time.
- Output one CSV per model plus a reusable `model_comparison.csv`.
- Display a text confusion matrix, so you don't need Matplotlib.

**Scope:** image-level compliance means **at least one visible worker clearly lacks a helmet or high-visibility vest**. The prompt asks about boots, but boots are *not scored* unless you have verified ground truth for boots. The model must not infer certified safety footwear just from appearance.


## Remote server models


In [4]:

from ollama import Client

# Connect to the remote Ollama server
client = Client(
    host="http://172.16.167.18:80",
    timeout=180
)

# List the models installed on the remote server
models = client.list()

for model in models.models:
    print(model.model)


llama3.1:latest
llava:13b
dlasher/MiniCPM-V-4.7-35B-A3B:IQ4_NL
qwen3.8:27b
all-minilm:33m
qwen3-embedding:latest
gemma4:26b
gpt-oss:20b
mxbai-embed-large:latest
gemma4:e4b


## 1. Install Ollama client (only if needed)


In [5]:
%pip install ollama

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 25.0.1 -> 26.3
[notice] To update, run: python.exe -m pip install --upgrade pip


## 2. Settings

Change `IMAGE_DIR` to the folder with your PPE images and `LABELS_CSV` to its ground-truth CSV. `RUN_LIMIT = 5` checks five **verified** images first. Set `0` to evaluate all verified images.

Use `MODE = "cloud"` only when your selected model is available through a configured hosted Ollama provider. Cloud processing sends your images to that provider; use only approved test images.


In [6]:
from pathlib import Path
from getpass import getpass
from ollama import Client
import csv
import json
import re
import time
from IPython.display import display

# -------------------------------------------------------------------------------------------------
# Choose local, remote or cloud model. Ctrl + / the model + mode you DON'T want to use.

# MODEL_CHOICE = "minicpm-v4.6:1b"
# MODE = "local"
# openbmb/minicpm-v4.7:1b, minicpm-v4.6:1b, gemma3:4b, llava:7b, qwen3-vl:8b (144s per image, takes forever to run)

MODEL_CHOICE = "llava:13b"  # 
MODE = "remote"                 
# qwen3.8:27b, dlasher/MiniCPM-V-4.7-35B-A3B:IQ4_NL, llava:13b, gemma4:26b, gemma4:e4b

# --------------------------------------------------------------------------------------------------

RUN_LIMIT = 0                 # 0 = evaluate all verified images
REQUEST_TIMEOUT = 300          # seconds per request
SHOW_IMAGES = False            # True = display test images during processing

# Select DATASET ID from the datasets array.
DATASET_ID = "construction_v1_60"  # construction_v1_60, mostly_vest_boots, with_boots

PROJECT_ROOT = Path.cwd()

DATASETS = {
    "construction_v1_60": {
        "images": PROJECT_ROOT / "model testing images",
        "labels": PROJECT_ROOT / "model testing images" / "ground_truth_review.csv",
    },
    "mostly_vest_boots": {
        "images": PROJECT_ROOT / "ppemostlyvestandboot" / "valid" / "images",
        "labels": PROJECT_ROOT / "ppemostlyvestandboot" / "ground_truth_review.csv",
    },
    "with_boots": {
        "images": PROJECT_ROOT / "ppewithboots" / "valid" / "images",
        "labels": PROJECT_ROOT / "ppewithboots" / "ground_truth.csv",
    },
}

# PPE_THINK_CHECKPOINT_V1
# Qwen: default, off, low, medium, high
# Gemma 4: default, off, on
# MiniCPM-V / LLaVA: default unless support is verified
THINK_MODE = "default"
NUM_PREDICT = 2048

# When False, even failed requests are not repeated on resume.
# Set True to retry failed images while retaining error history.
RETRY_ERRORS = False


if DATASET_ID not in DATASETS:
    raise ValueError(f"Unknown dataset: {DATASET_ID}")

IMAGE_DIR = DATASETS[DATASET_ID]["images"]
LABELS_CSV = DATASETS[DATASET_ID]["labels"]

# Every dataset uses its own reviewed ground-truth CSV.
if not LABELS_CSV.is_file():
    raise FileNotFoundError(
        f"Ground-truth CSV missing for {DATASET_ID}: {LABELS_CSV}. "
        "Place the reviewed CSV in the selected dataset folder."
    )


RESULTS_DIR = PROJECT_ROOT / "ppe_model_results" / DATASET_ID
PREDICTIONS_DIR = RESULTS_DIR / "predictions"
METRICS_DIR = RESULTS_DIR / "metrics"

PREDICTIONS_DIR.mkdir(parents=True, exist_ok=True)
METRICS_DIR.mkdir(parents=True, exist_ok=True)


if MODE == "remote":
    client = Client(
        host="http://172.16.167.18:80",
        timeout=REQUEST_TIMEOUT
    )

elif MODE == "cloud":
    api_key = getpass("Ollama Cloud API key: ")
    client = Client(
        host="https://ollama.com",
        headers={"Authorization": f"Bearer {api_key}"},
        timeout=REQUEST_TIMEOUT
    )

else:
    client = Client(
        host="http://localhost:11434",
        timeout=REQUEST_TIMEOUT
    )



print("Thinking:", THINK_MODE)
print("Max output tokens:", NUM_PREDICT)

print(f"Model: {MODEL_CHOICE} | Mode: {MODE}")
print(f"Images: {IMAGE_DIR}")
print(f"Labels: {LABELS_CSV}")


Thinking: default
Max output tokens: 2048
Model: llava:13b | Mode: remote
Images: c:\Users\GYS\Downloads\test model\model testing images
Labels: c:\Users\GYS\Downloads\test model\model testing images\ground_truth_review.csv


## 3. Prepare and verify ground truth

- `verified_helmet` and `verified_vest` use **yes** if one or more visible workers clearly lack that item, **no** if every assessable worker wears it, and **unclear** if you can't tell and no missing PPE is certain.
- `review_status` must be **verified** for a row to be scored.
- A dataset owner's annotation is a starting point; visually verify it. No missing annotation does **not** prove compliance.

If no CSV exists, this cell makes a **blank** review CSV. Open it in Excel, fill in the labels, and save it as CSV. It can also read the earlier review format with `verified_label` = `violation` / `compliant` / `unclear`.


In [7]:
if DATASET_ID in ("with_boots", "mostly_vest_boots"):
    if not LABELS_CSV.is_file():
        raise FileNotFoundError(
            f"Reviewed ground truth not found: {LABELS_CSV}"
        )

    with LABELS_CSV.open(
        newline="", encoding="utf-8-sig"
    ) as f:
        reader = csv.DictReader(f)
        labels = list(reader)
        headers = set(reader.fieldnames or [])

    required = {
        "source_dataset",
        "source_image",
        "verified_helmet",
        "verified_vest",
        "verified_boots",
        "review_status",
    }

    if not required <= headers:
        raise ValueError(
            f"Missing CSV columns: {sorted(required - headers)}"
        )

    source_filter = {
        "with_boots": "ppewithboots",
        "mostly_vest_boots": "ppemostlyvestandboot",
    }[DATASET_ID]

    allowed = {"yes", "no", "unclear"}
    selected_labels = []
    missing_images = []
    verified_in_dataset = 0
    data_type = "per_item"

    for record in labels:
        if record["source_dataset"].strip() != source_filter:
            continue

        if record["review_status"].strip().lower() != "verified":
            continue

        verified_in_dataset += 1
        row = dict(record)

        for item in ("helmet", "vest", "boots"):
            field = "verified_" + item
            row[field] = str(row[field]).strip().lower()

        if any(
            row["verified_" + item] not in allowed
            for item in ("helmet", "vest", "boots")
        ):
            raise ValueError(
                f"Invalid verified PPE labels: {row.get('review_id')}"
            )

        relative = Path(
            row["source_image"].replace("\\", "/")
        )

        if relative.is_absolute() or ".." in relative.parts:
            raise ValueError(f"Unsafe image path: {relative}")

        actual_path = PROJECT_ROOT / relative

        if not actual_path.is_file():
            missing_images.append(str(relative))
            continue

        row["_image_path"] = actual_path
        selected_labels.append(row)

    print("Dataset:", DATASET_ID)
    print("Ground truth:", LABELS_CSV)
    print("Verified rows in selected dataset:", verified_in_dataset)
    print("Verified image files found:", len(selected_labels))
    print("Missing image files:", len(missing_images))

    if missing_images:
        raise FileNotFoundError(
            "Some verified images were not found. "
            "First missing path: " + missing_images[0]
        )

    if not selected_labels:
        raise ValueError(
            "No verified images selected. Check source_dataset "
            "and source_image in ground_truth.csv."
        )

    for row in selected_labels[:3]:
        print(
            row["review_id"],
            row["_image_path"].name,
            row["verified_helmet"],
            row["verified_vest"],
            row["verified_boots"],
        )
else:
    from IPython.display import Image as NotebookImage

    if not IMAGE_DIR.is_dir():
        raise FileNotFoundError(f"Image folder not found: {IMAGE_DIR}. Edit IMAGE_DIR in Settings.")

    all_images = sorted(p for p in IMAGE_DIR.iterdir() if p.suffix.lower() in {'.jpg', '.jpeg', '.png'})
    if not all_images:
        raise ValueError(f"No JPG/PNG images found in: {IMAGE_DIR}")

    if not LABELS_CSV.exists():
        with LABELS_CSV.open('w', newline='', encoding='utf-8-sig') as f:
            writer = csv.DictWriter(f, fieldnames=['image', 'verified_helmet', 'verified_vest', 'review_status', 'notes'])
            writer.writeheader()
            writer.writerows({'image': p.name, 'verified_helmet': '', 'verified_vest': '',
                              'review_status': 'needs_review', 'notes': ''} for p in all_images)
        print(f"Blank review CSV created at: {LABELS_CSV}")
        print("Open this file in Excel, check your images and complete the labels first.")

    with LABELS_CSV.open(newline='', encoding='utf-8-sig') as f:
        reader = csv.DictReader(f)
        headers = set(reader.fieldnames or [])
        labels = list(reader)

    if not {'image', 'review_status'} <= headers:
        raise ValueError("Ground-truth CSV needs 'image' and 'review_status' columns.")

    if {'verified_helmet', 'verified_vest'} <= headers:
        data_type = 'per_item'
        allowed = {'yes', 'no', 'unclear'}
        selected_labels = [r for r in labels
            if r['review_status'].strip().lower() == 'verified'
            and str(r.get('verified_helmet','')).strip().lower() in allowed
            and str(r.get('verified_vest','')).strip().lower() in allowed]
    elif 'verified_label' in headers:
        data_type = 'overall_only'
        selected_labels = [r for r in labels
            if r['review_status'].strip().lower() == 'verified'
            and str(r.get('verified_label','')).strip().lower() in {'violation','compliant','unclear'}]
    else:
        raise ValueError("CSV needs either verified_helmet + verified_vest, or verified_label.")

    for row in selected_labels:
        for field in ('verified_helmet', 'verified_vest', 'verified_label'):
            if field in row:
                row[field] = str(row[field]).strip().lower()
    selected_labels = [r for r in selected_labels if (IMAGE_DIR / r['image']).is_file()]

    print(f"Found {len(all_images)} images, {len(labels)} CSV entries, {len(selected_labels)} verified images.")
    print("Ground-truth format:", data_type)
    if not selected_labels:
        print("No verified images yet. Check the images, update the CSV and rerun this cell.")
    else:
        for row in selected_labels[:6]:
            print(' ', row['image'], '|', row.get('verified_helmet',''), row.get('verified_vest',''), row.get('verified_label',''))


Found 60 images, 60 CSV entries, 60 verified images.
Ground-truth format: per_item
  000005_jpg.rf.c613c066b79795d219855aca269760e6.jpg | no yes 
  006672_jpg.rf.657b0a9ecd720d07ca5dd75338a058d5.jpg | no no 
  002551_jpg.rf.b09484b130c1587abfa8365c3d774bf1.jpg | no yes 
  IMG_5846_jpg.rf.f82d7ebbe6b6c050cacb8a7e0ff64424.jpg | no no 
  003357_jpg.rf.9fb1f2cebc239b4276a81ce2cbf9e911.jpg | no yes 
  image_53_jpg.rf.126c05efe93729b667ca2860e6fd0ac3.jpg | no no 


## 4. Prompt and parsing helpers

For fair comparisons, use exactly the same image and prompt for every model. The prompt requests JSON so Python can measure *actual predictions* rather than guess from free-form text. The script considers an invalid response an `error`, not a compliant image.

In [8]:
PROMPT = """Assess this construction-site image for PPE compliance.
Inspect every visible worker for a safety helmet and a high-visibility vest.
Also describe visible safety boots, but do NOT infer certified safety footwear from appearance alone.

For helmet/vest classifications: yes = at least one worker is clearly missing it;
no = all visible workers can be assessed and none are missing it;
unclear = at least one worker cannot be assessed, and no definite absence is visible.
Never guess when the relevant body part is obstructed.

Return ONLY valid JSON, with exactly these fields and permitted values:
{
  "worker_count": 2,
  "workers": [
    {"position": "left", "helmet": "worn", "vest": "missing", "boots": "unclear"}
  ],
  "helmet_violation": "no",
  "vest_violation": "yes",
  "boots_violation": "unclear"
}
For every violation field, choose only one of: yes, no, unclear. The JSON above is an example of formatting, NOT the expected answer.
"""

def overall_from_items(helmet, vest, boots=None):
    """Image-level decision from helmet/vest only (boots not in ground truth)."""
    if 'yes' in (helmet, vest, boots):
        return 'violation'
    if helmet == 'no' and vest == 'no' and (boots is None or boots == 'no'):
        return 'compliant'
    return 'unclear'

def parse_vlm_response(raw):
    """Parse valid JSON; explicitly report noncompliant/unparseable responses."""
    raw = str(raw or '').strip()
    raw = re.sub(r'^```(?:json)?\s*|\s*```$', '', raw, flags=re.I)
    try:
        obj = json.loads(raw)
    except (ValueError, TypeError):
        left, right = raw.find('{'), raw.rfind('}')
        if left < 0 or right <= left:
            return None
        try:
            obj = json.loads(raw[left:right + 1])
        except (ValueError, TypeError):
            return None
    if not isinstance(obj, dict):
        return None
    for field in ('helmet_violation', 'vest_violation'):
        v = obj.get(field)
        if not isinstance(v, str) or v.strip().lower() not in ('yes', 'no', 'unclear'):
            return None
        obj[field] = v.strip().lower()
    boots = str(obj.get('boots_violation', '')).strip().lower()
    if boots not in ('yes', 'no', 'unclear'):
        return None
    obj['boots_violation'] = boots
    return obj

def truth_for_row(row):
    if data_type == 'per_item':
        h, v = row['verified_helmet'], row['verified_vest']
        return h, v, overall_from_items(h, v, row.get('verified_boots'))
    overall = row['verified_label']
    return '', '', overall

# Small offline sanity checks: no Ollama call or model download.
assert overall_from_items('no','no') == 'compliant'
assert overall_from_items('yes','unclear') == 'violation'
assert overall_from_items('no','unclear') == 'unclear'
assert parse_vlm_response('{"helmet_violation":"yes","vest_violation":"no","boots_violation":"unclear"}') is not None
print('Parsing and ground-truth helpers are ready.')

Parsing and ground-truth helpers are ready.


# Running the Model

## 5. Analyse verified images with the selected model

Each image is analysed separately. Model failures or non-JSON responses are marked as `error`—not silently interpreted as compliant. We time every request including any network latency.


In [9]:

import hashlib
import inspect
import os

if not selected_labels:
    raise ValueError(
        "No verified images selected. Run Section 3 first."
    )

allowed_modes = {"default"}

if MODEL_CHOICE.startswith("qwen3.8:"):
    allowed_modes = {"default", "off", "low", "medium", "high"}
elif MODEL_CHOICE.startswith("gemma4:"):
    allowed_modes = {"default", "off", "on"}

if THINK_MODE not in allowed_modes:
    raise ValueError(
        f"{MODEL_CHOICE} supports these configured thinking modes: "
        f"{sorted(allowed_modes)}. Change THINK_MODE in Section 2."
    )

if NUM_PREDICT is not None and NUM_PREDICT < 1:
    raise ValueError("NUM_PREDICT must be positive or None.")

prompt_sha = hashlib.sha256(
    PROMPT.encode("utf-8")
).hexdigest()[:12]

labels_sha = hashlib.sha256(
    LABELS_CSV.read_bytes()
).hexdigest()[:12]

configuration = {
    "model_id": MODEL_CHOICE,
    "dataset": DATASET_ID,
    "thinking_mode": THINK_MODE,
    "num_predict": NUM_PREDICT,
    "temperature": 0,
    "timeout_seconds": REQUEST_TIMEOUT,
    "prompt_sha": prompt_sha,
    "ground_truth_sha": labels_sha,
}

config_hash = hashlib.sha256(
    json.dumps(configuration, sort_keys=True).encode("utf-8")
).hexdigest()[:12]

safe_model = re.sub(
    r"[^a-zA-Z0-9_-]+", "_", MODEL_CHOICE
)[:24]

CONFIG_ID = f"{safe_model}_{config_hash}"

CONFIG_LABEL = (
    f"{MODEL_CHOICE} [think={THINK_MODE}, "
    f"max={NUM_PREDICT}, prompt={prompt_sha[:6]}]"
)

checkpoint_dir = RESULTS_DIR / "checkpoints"
checkpoint_dir.mkdir(parents=True, exist_ok=True)

checkpoint_path = checkpoint_dir / f"{CONFIG_ID}.csv"
manifest_path = checkpoint_dir / f"{CONFIG_ID}.json"

manifest_path.write_text(
    json.dumps(configuration, indent=2),
    encoding="utf-8",
)

def image_key(row):
    return str(row.get("source_image") or row["image"])

def save_checkpoint(records):
    rows = list(records.values())
    if not rows:
        return

    columns = list(dict.fromkeys(
        field
        for record in rows
        for field in record.keys()
    ))

    temp_path = checkpoint_path.with_suffix(".csv.tmp")

    with temp_path.open(
        "w", newline="", encoding="utf-8-sig"
    ) as f:
        writer = csv.DictWriter(
            f, fieldnames=columns, extrasaction="ignore"
        )
        writer.writeheader()
        writer.writerows(rows)
        f.flush()
        os.fsync(f.fileno())

    os.replace(temp_path, checkpoint_path)

completed = {}

if checkpoint_path.exists():
    with checkpoint_path.open(
        newline="", encoding="utf-8-sig"
    ) as f:
        for saved in csv.DictReader(f):
            saved["seconds"] = float(saved.get("seconds") or 0)
            completed[image_key(saved)] = saved

run_rows = (
    selected_labels[:RUN_LIMIT]
    if RUN_LIMIT > 0
    else selected_labels
)

print("Configuration:", CONFIG_LABEL)
print("Verified images:", len(selected_labels))
print("Target images:", len(run_rows))
print("Previously saved results:", len(completed))
print("Checkpoint:", checkpoint_path)

run_started = time.perf_counter()
new_requests = 0
skipped_requests = 0

for i, row in enumerate(run_rows, start=1):
    key = image_key(row)
    previous = completed.get(key)

    if previous is not None:
        if previous.get("predicted") != "error" or not RETRY_ERRORS:
            skipped_requests += 1
            print(f"[{i}/{len(run_rows)}] Resumed: {key}")
            continue

    path = row.get("_image_path")
    if path is None:
        if row.get("source_image"):
            path = PROJECT_ROOT / row["source_image"]
        else:
            path = IMAGE_DIR / row["image"]

    path = Path(path)

    if not path.is_file():
        raise FileNotFoundError(f"Image not found: {path}")

    truth_h, truth_v, truth_overall = truth_for_row(row)

    print(f"[{i}/{len(run_rows)}] Analysing: {path.name}")

    if SHOW_IMAGES:
        display(NotebookImage(filename=str(path), width=320))

    request = {
        "model": MODEL_CHOICE,
        "messages": [{
            "role": "user",
            "content": PROMPT,
            "images": [path.read_bytes()],
        }],
        "format": "json",
        "stream": False,
        "options": {"temperature": 0},
    }

    if NUM_PREDICT is not None:
        request["options"]["num_predict"] = NUM_PREDICT

    if THINK_MODE == "off":
        request["think"] = False
    elif THINK_MODE == "on":
        request["think"] = True
    elif THINK_MODE in ("low", "medium", "high"):
        request["think"] = THINK_MODE

    started = time.perf_counter()
    raw, parsed, error = "", None, ""
    eval_count, thinking_chars, done_reason = "", "", ""

    try:
        response = client.chat(**request)

        raw = response.message.content or ""
        parsed = parse_vlm_response(raw)

        eval_count = response.eval_count or 0
        thinking_chars = len(response.message.thinking or "")
        done_reason = response.done_reason or ""

        if parsed is None:
            error = "Invalid or incomplete PPE JSON"

        if done_reason == "length":
            error = "Output token limit reached"
            parsed = None

    except Exception as exc:
        error = f"{type(exc).__name__}: {exc}"

    seconds = round(time.perf_counter() - started, 3)

    predicted_h = (
        parsed["helmet_violation"] if parsed else "error"
    )
    predicted_v = (
        parsed["vest_violation"] if parsed else "error"
    )
    predicted_b = (
        parsed["boots_violation"] if parsed else "error"
    )

    if parsed:
        if len(inspect.signature(overall_from_items).parameters) >= 3:
            predicted_o = overall_from_items(
                predicted_h, predicted_v, predicted_b
            )
        else:
            predicted_o = overall_from_items(
                predicted_h, predicted_v
            )
    else:
        predicted_o = "error"

    error_history = []
    attempts = 1

    if previous:
        attempts = int(previous.get("attempts") or 1) + 1
        try:
            error_history = json.loads(
                previous.get("previous_errors") or "[]"
            )
        except ValueError:
            error_history = []

        if previous.get("error"):
            error_history.append(previous["error"])

    result = {
        "image": row["image"],
        "source_image": row.get("source_image", ""),
        "model": CONFIG_LABEL,
        "model_id": MODEL_CHOICE,
        "thinking_mode": THINK_MODE,
        "num_predict": (
            NUM_PREDICT if NUM_PREDICT is not None else ""
        ),
        "prompt_sha": prompt_sha,
        "ground_truth": truth_overall,
        "predicted": predicted_o,
        "truth_helmet": truth_h,
        "pred_helmet": predicted_h,
        "truth_vest": truth_v,
        "pred_vest": predicted_v,
        "truth_boots": row.get("verified_boots", ""),
        "pred_boots": predicted_b,
        "pred_worker_count": (
            parsed.get("worker_count", "") if parsed else ""
        ),
        "seconds": seconds,
        "eval_count": eval_count,
        "thinking_characters": thinking_chars,
        "done_reason": done_reason,
        "attempts": attempts,
        "previous_errors": json.dumps(error_history),
        "error": error,
        "raw_response": raw,
    }

    completed[key] = result

    # Save immediately after each image, including failed requests.
    save_checkpoint(completed)
    new_requests += 1

    print(
        f"  Truth: {truth_overall} | "
        f"Prediction: {predicted_o} | {seconds:.2f}s"
    )

    if error:
        print("  ERROR:", error)

# Reconstruct the full set from checkpoints, including resumed rows.
# This also preserves earlier results if RUN_LIMIT is reduced.
predictions = [
    completed[image_key(row)]
    for row in selected_labels
    if image_key(row) in completed
]

total_seconds = time.perf_counter() - run_started

print("\nEvaluation run finished.")
print("New requests:", new_requests)
print("Reused previous results:", skipped_requests)
print("Total saved predictions:", len(predictions))
print("Elapsed this session:", round(total_seconds, 2), "seconds")
print("Checkpoint:", checkpoint_path)


Configuration: llava:13b [think=default, max=2048, prompt=47e670]
Verified images: 60
Target images: 60
Previously saved results: 60
Checkpoint: c:\Users\GYS\Downloads\test model\ppe_model_results\construction_v1_60\checkpoints\llava_13b_918a9ff41036.csv
[1/60] Resumed: 000005_jpg.rf.c613c066b79795d219855aca269760e6.jpg
[2/60] Resumed: 006672_jpg.rf.657b0a9ecd720d07ca5dd75338a058d5.jpg
[3/60] Resumed: 002551_jpg.rf.b09484b130c1587abfa8365c3d774bf1.jpg
[4/60] Resumed: IMG_5846_jpg.rf.f82d7ebbe6b6c050cacb8a7e0ff64424.jpg
[5/60] Resumed: 003357_jpg.rf.9fb1f2cebc239b4276a81ce2cbf9e911.jpg
[6/60] Resumed: image_53_jpg.rf.126c05efe93729b667ca2860e6fd0ac3.jpg
[7/60] Resumed: 004063_jpg.rf.6ecc587c20a87ec176841aea39b8f5db.jpg
[8/60] Resumed: image_55_jpg.rf.51b5b121599680f79fff1ad2d6aefba0.jpg
[9/60] Resumed: 004763_jpg.rf.c8f1bd1cb7cf6037f7c90a6c213d6364.jpg
[10/60] Resumed: ppe_1073_jpg.rf.92536fa1d42874966bc7622b6f87290a.jpg
[11/60] Resumed: youtube-114_jpg.rf.eb3011bd70cacf2dc16e547a76bf46

In [10]:
from pathlib import Path
from collections import Counter
import csv

print("Working directory:", Path.cwd())
print("Selected dataset:", globals().get("DATASET_ID"))
print("Ground truth path:", globals().get("LABELS_CSV"))
print("Selected labels:", len(globals().get("selected_labels", [])))

csv_path = Path(globals().get("LABELS_CSV", "ground_truth.csv"))

if csv_path.is_file():
    with csv_path.open(encoding="utf-8-sig", newline="") as f:
        rows = list(csv.DictReader(f))

    print("CSV rows:", len(rows))
    print("Review statuses:", dict(Counter(
        r.get("review_status", "").strip().lower()
        for r in rows
    )))

    verified = [
        r for r in rows
        if r.get("review_status", "").strip().lower() == "verified"
    ]

    print("Verified in CSV:", len(verified))

    for row in verified[:3]:
        image = Path(row.get("source_image") or row["image"])
        actual = PROJECT_ROOT / image if row.get("source_image") else IMAGE_DIR / image
        print("Image exists:", actual.is_file(), image)
else:
    print("ERROR: Ground-truth CSV not found.")

Working directory: c:\Users\GYS\Downloads\test model
Selected dataset: construction_v1_60
Ground truth path: c:\Users\GYS\Downloads\test model\model testing images\ground_truth_review.csv
Selected labels: 60
CSV rows: 60
Review statuses: {'verified': 60}
Verified in CSV: 60
Image exists: True 000005_jpg.rf.c613c066b79795d219855aca269760e6.jpg
Image exists: True 006672_jpg.rf.657b0a9ecd720d07ca5dd75338a058d5.jpg
Image exists: True 002551_jpg.rf.b09484b130c1587abfa8365c3d774bf1.jpg


### 5A. Analyse response reliability (errors vs uncertain classifications)

A parseable `unclear` response is **not** a request failure. A model can run successfully but produce unusable JSON; we count those as errors.


In [11]:
from collections import Counter

outcomes = Counter(row['predicted'] for row in predictions)

image_count = len(predictions)
decisive_count = outcomes['violation'] + outcomes['compliant']
unclear_count = outcomes['unclear']
error_count = outcomes['error']
valid_response_count = image_count - error_count

print(f"Images attempted:                 {image_count}")
print(f"Valid, parseable classifications: {valid_response_count}")
print(f"Decisive predictions:             {decisive_count}")
print(f"Unclear predictions:              {unclear_count}")
print(f"Unsuccessful analyses (errors):   {error_count}")

empty_response_errors = sum(
    r['predicted'] == 'error'
    and r['error'].startswith('Unparseable JSON')
    and not (r['raw_response'] or '').strip()
    for r in predictions
)

print(f"Empty model responses:            {empty_response_errors}")
print(f"Other format/request errors:      {error_count - empty_response_errors}")

if error_count:
    print("\nFiles requiring error review:")
    for r in predictions:
        if r['predicted'] == 'error':
            print(f"- {r['image']}: {r['error']}")

Images attempted:                 60
Valid, parseable classifications: 60
Decisive predictions:             60
Unclear predictions:              0
Unsuccessful analyses (errors):   0
Empty model responses:            0
Other format/request errors:      0


troubleshooting

In [12]:
# if prediction had error this will have output of the image name, error message and the raw response from the model
for result in predictions:
    if result["predicted"] == "error":
        print("Image:", result["image"])
        print("Error:", result["error"])
        print("\nOriginal model response:")
        print(repr(result["raw_response"]))

## 6. Evaluate accuracy, precision, recall, F1 and the confusion matrix

Scores are computed on **binary, decisive** predictions, with separate measures for cases where the model responds `unclear` or fails:

- **Coverage:** fraction of verified binary images yielding a decisive `violation` or `compliant` result.
- **Strict accuracy:** correct predictions divided by *all* verified binary images (unclear/errors count as not correct).
- **Actionable recall:** correctly detected violations divided by *all* confirmed violations (unclear/errors count as missed for this measure).

A class absent from your test set makes some metrics **undefined**, not zero. The text confusion matrix uses the usual values TN, FP, FN, TP.


In [13]:
def safe_div(numerator, denominator):
    return numerator / denominator if denominator else None

def metric_report(rows, truth_col, prediction_col, positive='violation', negative='compliant'):
    labeled = [r for r in rows if r[truth_col] in (positive, negative)]
    decided = [r for r in labeled if r[prediction_col] in (positive, negative)]
    errors = sum(r[prediction_col] == 'error' for r in labeled)
    unclear = sum(r[prediction_col] == 'unclear' for r in labeled)
    tp = sum(r[truth_col] == positive and r[prediction_col] == positive for r in decided)
    tn = sum(r[truth_col] == negative and r[prediction_col] == negative for r in decided)
    fp = sum(r[truth_col] == negative and r[prediction_col] == positive for r in decided)
    fn = sum(r[truth_col] == positive and r[prediction_col] == negative for r in decided)
    total_positive = sum(r[truth_col] == positive for r in labeled)
    return {
        'ground_truth_binary': len(labeled),
        'valid_responses': len(labeled) - errors,
        'error_predictions': errors,
        'unclear_predictions': unclear,
        'scored_decisive': len(decided),
        'unclear_or_error_on_binary': len(labeled) - len(decided),
        'tp': tp, 'tn': tn, 'fp': fp, 'fn': fn,
        'accuracy': safe_div(tp + tn, len(decided)),
        'precision': safe_div(tp, tp + fp),
        'recall': safe_div(tp, tp + fn),
        'f1': safe_div(2 * tp, 2 * tp + fp + fn),
        'false_positive_rate': safe_div(fp, fp + tn),
        'coverage': safe_div(len(decided), len(labeled)),
        'strict_accuracy': safe_div(tp + tn, len(labeled)),
        'actionable_recall': safe_div(tp, total_positive),
    }

# Unit checks using synthetic outcomes, NOT actual PPE model scores
_test = [
    {'gold':'violation','pred':'violation'},
    {'gold':'compliant','pred':'violation'},
    {'gold':'violation','pred':'unclear'},
    {'gold':'compliant','pred':'compliant'}
]
_check = metric_report(_test, 'gold', 'pred')
assert (_check['tp'], _check['tn'], _check['fp'], _check['fn']) == (1,1,1,0)
assert abs(_check['strict_accuracy'] - 0.5) < 1e-9
assert abs(_check['actionable_recall'] - 0.5) < 1e-9

reports = {'overall': metric_report(predictions, 'ground_truth', 'predicted')}
if data_type == 'per_item':
    reports['helmet'] = metric_report(predictions, 'truth_helmet', 'pred_helmet', 'yes', 'no')
    reports['vest'] = metric_report(predictions, 'truth_vest', 'pred_vest', 'yes', 'no')
    if predictions and predictions[0].get('truth_boots'):
        reports['boots'] = metric_report(
            predictions, 'truth_boots', 'pred_boots', 'yes', 'no'
        )

average_seconds = safe_div(sum(r['seconds'] for r in predictions), len(predictions))
print(f"Model: {MODEL_CHOICE} | Images: {len(predictions)}")
print(f"Average seconds/image: {average_seconds:.2f}" if average_seconds is not None else "No timing data")
print(f"Total model-request seconds: {sum(r['seconds'] for r in predictions):.2f}")
for scope, report in reports.items():
    print(f"\n--- {scope.upper()} ---")
    for key in ('ground_truth_binary','valid_responses','error_predictions','unclear_predictions','scored_decisive',
                'tp','tn','fp','fn','accuracy','precision','recall','f1','false_positive_rate','coverage','strict_accuracy','actionable_recall'):
        val = report[key]
        shown = 'undefined' if val is None else f'{val:.3f}' if isinstance(val, float) else val
        print(f'{key:27} {shown}')

r = reports['overall']
print('\nCONFUSION MATRIX (only decisive cases, overall)')
print('                         Predicted compliant   Predicted violation')
print(f"Actual compliant         {r['tn']:>19}   {r['fp']:>19}")
print(f"Actual violation         {r['fn']:>19}   {r['tp']:>19}")


Model: llava:13b | Images: 60
Average seconds/image: 3.92
Total model-request seconds: 235.21

--- OVERALL ---
ground_truth_binary         60
valid_responses             60
error_predictions           0
unclear_predictions         0
scored_decisive             60
tp                          29
tn                          0
fp                          31
fn                          0
accuracy                    0.483
precision                   0.483
recall                      1.000
f1                          0.652
false_positive_rate         1.000
coverage                    1.000
strict_accuracy             0.483
actionable_recall           1.000

--- HELMET ---
ground_truth_binary         59
valid_responses             59
error_predictions           0
unclear_predictions         0
scored_decisive             59
tp                          0
tn                          40
fp                          0
fn                          19
accuracy                    0.678
precision        

## 7. Export predictions, metrics and model comparisons

Exports standard CSV files that Excel can open. A model's comparison rows are replaced when you rerun it; other models' rows are retained. No Pandas needed.


In [14]:
def write_csv(path, rows, columns):
    with path.open('w', newline='', encoding='utf-8-sig') as f:
        writer = csv.DictWriter(f, fieldnames=columns, extrasaction='ignore')
        writer.writeheader()
        writer.writerows(rows)

safe_name = CONFIG_ID
results_path = PREDICTIONS_DIR / f'predictions_{safe_name}.csv'
metrics_path = METRICS_DIR / f'metrics_{safe_name}.csv'
comparison_path = RESULTS_DIR / 'model_comparison.csv'

prediction_columns = list(predictions[0].keys()) if predictions else []
metric_columns = list(next(iter(reports.values())).keys())
write_csv(results_path, predictions, prediction_columns)
write_csv(metrics_path, [{'scope': scope, **m} for scope, m in reports.items()], ['scope'] + metric_columns)

# Group response reliability metrics together
reliability_columns = [
    'model',
    'model_id',
    'thinking_mode',
    'num_predict',
    'prompt_sha',
    'scope',
    'images_analyzed',
    'valid_responses',
    'error_predictions',
    'unclear_predictions',
    'scored_decisive',
    'coverage'
]

comparison_columns = reliability_columns + [
    'avg_seconds',
    'total_request_seconds'
] + [
    col for col in metric_columns
    if col not in reliability_columns
]
new_rows = [{
    'model': CONFIG_LABEL,
    'model_id': MODEL_CHOICE,
    'thinking_mode': THINK_MODE,
    'num_predict': NUM_PREDICT if NUM_PREDICT is not None else '',
    'prompt_sha': prompt_sha,
    'scope': scope,
    'images_analyzed': len(predictions),
    'avg_seconds': round(average_seconds, 3) if average_seconds is not None else '',
    'total_request_seconds': round(sum(r['seconds'] for r in predictions), 3),
    **m,
} for scope, m in reports.items()]

existing = []
if comparison_path.exists():
    with comparison_path.open(newline='', encoding='utf-8-sig') as f:
        existing = list(csv.DictReader(f))
existing = [r for r in existing if not (r.get('model') == CONFIG_LABEL and r.get('scope') in reports)]
comparison = existing + new_rows
write_csv(comparison_path, comparison, comparison_columns)

print('Saved:', results_path)
print('Saved:', metrics_path)
print('Updated:', comparison_path)
print('\nModel comparison:')
for r in comparison:
    print(f"  {r['model']:28} {r['scope']:8} accuracy={r['accuracy']} recall={r['recall']} f1={r['f1']} avg_seconds={r['avg_seconds']}")


Saved: c:\Users\GYS\Downloads\test model\ppe_model_results\construction_v1_60\predictions\predictions_llava_13b_918a9ff41036.csv
Saved: c:\Users\GYS\Downloads\test model\ppe_model_results\construction_v1_60\metrics\metrics_llava_13b_918a9ff41036.csv
Updated: c:\Users\GYS\Downloads\test model\ppe_model_results\construction_v1_60\model_comparison.csv

Model comparison:
  gemma3:4b                    overall  accuracy=0.55 recall=1.0 f1=0.6823529411764706 avg_seconds=2.707
  gemma3:4b                    helmet   accuracy=0.6779661016949152 recall=0.0 f1=0.0 avg_seconds=2.707
  gemma3:4b                    vest     accuracy=0.5166666666666667 recall=1.0 f1=0.6506024096385542 avg_seconds=2.707
  llava:7b                     overall  accuracy=0.48333333333333334 recall=0.9655172413793104 f1=0.6436781609195402 avg_seconds=2.273
  llava:7b                     helmet   accuracy=0.6779661016949152 recall=0.0 f1=0.0 avg_seconds=2.273
  llava:7b                     vest     accuracy=0.45 recall=0.

### Interpreting your results

- If your sample contains only violations, it cannot measure the false-positive rate. Include genuine compliant images.
- If there are no manually verified labels, accuracy cannot be calculated responsibly.
- `unclear` and invalid responses reduce coverage and strict accuracy even if decisive-only accuracy is high.
- The first model request may be slower because Ollama needs to load the weights into memory; keep the same conditions across tests.
- For all commercial/remote cloud use, ensure your image data is permitted to leave your computer.
- This notebook scores **helmet and vest** only. Boots require separate verified boot labels.
